# PRIORITEST-STAR — entraînement complet sur Google Colab

Priorisation des classes Java à risque de défaut : **TCN (historique) + GraphSAGE (imports) + fusion guidée par similarité + InfoNCE inter-vues + RankNet**.

**Protocole** : Ant / JEdit → entraînement sur la 1re version cible, réglages sur la 2e (validation), test final sur la 3e.
Il s'agit d'une **détection des défauts de la version observée**, pas d'une prédiction de la version suivante.

👉 *Exécution → Modifier le type d'exécution → GPU (T4)* est optionnel : le modèle est petit, le CPU suffit.

## 1. Récupérer le projet

In [ ]:
import os, pathlib, subprocess
REPO = "https://github.com/mohamedaoui/prioritest-star.git"   # dépôt privé : https://<TOKEN>@github.com/mohamedaoui/prioritest-star.git
if not pathlib.Path("/content/prioritest-star").exists():
    subprocess.run(["git", "clone", "-q", REPO, "/content/prioritest-star"], check=True)
else:  # session déjà utilisée : récupérer la dernière version du code
    subprocess.run(["git", "-C", "/content/prioritest-star", "pull", "-q", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", "/content/prioritest-star", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
os.chdir("/content/prioritest-star/KIT_MULTI_PROJETS")
print("Dossier de travail :", os.getcwd())

*Alternative sans GitHub* : zipper le dossier `capston`, puis exécuter la cellule suivante (sinon, l'ignorer).

In [ ]:
# from google.colab import files; import zipfile
# up = files.upload(); z = next(k for k in up if k.endswith('.zip'))
# zipfile.ZipFile(z).extractall('/content/proj')
# KIT = next(pathlib.Path('/content/proj').rglob('run_star.py')).parent; os.chdir(KIT); print(os.getcwd())

## 2. Dépendances, tests unitaires et contrôle des données

In [ ]:
!pip install -q -r requirements.txt
!python -m unittest discover -s tests -v
!python check_graph_temporal.py

In [ ]:
import torch, numpy as np
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device :", DEVICE)
for v in ["1.5", "1.6", "1.7"]:
    d = np.load(f"data/ant/ant-{v}.npz")
    print(f"ant-{v}: X{d['X'].shape} edges{d['edges'].shape} défauts={int(d['y'].sum())}/{len(d['y'])}")

## 3. Ant : 7 configurations × 5 graines (réglage des hyperparamètres sur la validation)

| # | Configuration | Question isolée |
|---|---|---|
| A1 | Boosting (métriques courantes) | référence tabulaire forte |
| A2 | TCN + BCE | apport de l'historique seul |
| A3 | + graphe + fusion classique | apport du graphe |
| A4 | + fusion guidée par similarité | apport de l'accord cosinus |
| A5 | + InfoNCE inter-vues | apport de l'alignement |
| A6 | + RankNet = **STAR complet** | apport du ranking |
| A7 | **STAR+** : Δ-métriques, graphe bidirectionnel, InfoNCE sans faux négatifs, RankNet sensible au LOC | amélioration proposée |

In [ ]:
!python run_star.py --project ant --seeds 42 43 44 45 46 --epochs 60 --tune --device {DEVICE}

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
def show(project):
    s = pd.read_csv(f"results/{project}/summary.csv")
    cols = ["variant"] + [c for c in s.columns if c.startswith("test_") and c.endswith(("_mean", "_std"))] + ["training_seconds_mean"]
    return s[cols]
show("ant")

## 4. Courbe « défauts retrouvés vs budget d'inspection (LOC) »

In [ ]:
import matplotlib.pyplot as plt, glob
def budget_curve(df, density=False):
    # density=True : tri par risque / LOC (classement sensible à l'effort)
    df = df.copy()
    df["_density"] = df["score"] / df["loc"].clip(lower=1)   # calculé ici : marche aussi avec d'anciens CSV
    df = df.sort_values("_density" if density else "score", ascending=False, kind="stable")
    x = np.concatenate([[0], df["loc"].clip(lower=1).cumsum() / df["loc"].clip(lower=1).sum()])
    y = np.concatenate([[0], df["label"].cumsum() / max(1, df["label"].sum())])
    return x, y
def plot_budget(project, seed=42):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, density in zip(axes, (False, True)):
        for f in sorted(glob.glob(f"results/{project}/seed{seed}/*_scores.csv")):
            d = pd.read_csv(f); d = d[d.split == "test"]
            x, y = budget_curve(d, density); ax.plot(x, y, label=pathlib.Path(f).stem.replace("_scores", ""))
        ax.plot([0, 1], [0, 1], ls=":", c="k", lw=.8, label="aléatoire")
        ax.axvline(.2, ls="--", c="grey"); ax.set_xlabel("Fraction du LOC inspecté"); ax.grid(alpha=.3)
        ax.set_title(f"{project} — test, seed {seed} — tri par " + ("risque / LOC" if density else "risque"))
    axes[0].set_ylabel("Fraction des classes défectueuses trouvées"); axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()
plot_budget("ant")

**Classement sensible à l'effort (risque / LOC).** Le risque de défaut est corrélé à la taille : trier par risque seul
place d'abord les grosses classes, qui épuisent le budget d'inspection. Trier par *densité de risque* (score ÷ LOC)
inspecte d'abord les classes au plus fort risque **par ligne**, ce qui maximise les défauts trouvés à budget LOC fixé
(principe des modèles *effort-aware* : Mende & Koschke 2010, Kamei et al. 2013). Colonne `test_EA_Recall@20pct_LOC_*`.

In [ ]:
s = pd.read_csv("results/ant/summary.csv")
s[["variant", "test_Recall@20pct_LOC_mean", "test_EA_Recall@20pct_LOC_mean", "test_EA_Recall@20pct_LOC_std"]]

## 5. Analyse : poids de fusion α/β et accord temporel–structurel

In [ ]:
d = pd.read_csv("results/ant/seed42/A6_STAR_full_scores.csv"); d = d[d.split == "test"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lab, g in d.groupby("label"):
    ax[0].hist(g.agreement_cos, bins=25, alpha=.6, label=f"label={lab}")
    ax[1].hist(g.alpha_temporal, bins=25, alpha=.6, label=f"label={lab}")
ax[0].set_title("cos(zT, zS)"); ax[1].set_title("alpha (poids vue temporelle)"); ax[0].legend(); ax[1].legend(); plt.show()
print("Top-15 des classes prioritaires (test) :")
d.sort_values("score", ascending=False).head(15)[["class_id", "label", "loc", "score", "alpha_temporal", "agreement_cos"]]

In [ ]:
# Erreurs : classes saines classées tout en haut / classes défectueuses classées tout en bas
r = d.sort_values("score", ascending=False).reset_index(drop=True)
print("Faux positifs du Top-20 :"); display(r.head(20).query("label == 0")[["class_id", "loc", "score", "agreement_cos"]])
print("Défauts manqués (bas du classement) :"); display(r.tail(40).query("label == 1")[["class_id", "loc", "score", "agreement_cos"]])

## 6. Deuxième projet : JEdit (3 fenêtres), puis Camel / POI (exploratoire, 2 fenêtres)

In [ ]:
!python run_star.py --project jedit --seeds 42 43 44 45 46 --epochs 60 --tune --device {DEVICE}
show("jedit")

In [ ]:
plot_budget("jedit")

In [ ]:
# Exploratoire : pas de validation => pas de --tune, nombre d'époques fixe
!python run_star.py --project camel --seeds 42 43 44 45 46 --epochs 40 --device {DEVICE}
!python run_star.py --project poi   --seeds 42 43 44 45 46 --epochs 40 --device {DEVICE}
display(show("camel")); display(show("poi"))

## 7. Version améliorée (`--improved`) et accuracy

Appliqué à **toutes** les variantes (comparaison équitable), sans jamais utiliser le test pour régler quoi que ce soit :
1. **Réentraînement sur entraînement + validation** (Ant 1.5+1.6 → test 1.7) avec l'époque et les hyperparamètres choisis sur la validation ;
2. **pos_weight** dans la BCE (classes défectueuses minoritaires) ;
3. **compression log** des métriques à queue lourde.

Toujours actifs : **règle de décision réglée sur la validation** (fraction de classes signalées) → Accuracy, Balanced accuracy, MCC, F1 ; **A8 = ensemble** boosting + A5 + A6.

⚠ Toujours comparer l'accuracy à `Majority_accuracy` (prédire « aucun défaut ») : sur JEdit 4.3, 97 % est obtenu sans rien détecter.

In [ ]:
for p, extra in [("ant", "--tune --epochs 60"), ("jedit", "--tune --epochs 60"), ("camel", "--epochs 40"), ("poi", "--epochs 40")]:
    !python run_star.py --project {p} --seeds 42 43 44 45 46 {extra} --improved --device {DEVICE}

In [ ]:
cols = ["test_Accuracy_mean", "test_Accuracy_std", "test_Majority_accuracy_mean", "test_Balanced_accuracy_mean",
        "test_MCC_mean", "test_AP_mean", "test_EA_Recall@20pct_LOC_mean"]
for p in ["ant", "jedit", "camel", "poi"]:
    base = pd.read_csv(f"results/{p}/summary.csv").set_index("variant")
    imp = pd.read_csv(f"results/{p}_improved/summary.csv").set_index("variant")
    print(f"\n===== {p} : protocole de base  vs  --improved =====")
    display(pd.concat({"base": base.reindex(columns=cols), "improved": imp.reindex(columns=cols)}, axis=1).round(3))

## 8. Sauvegarder les résultats

In [ ]:
!zip -qr /content/results_prioritest.zip results
from google.colab import files; files.download("/content/results_prioritest.zip")